# Análisis de Ventas de Distribuidora

## 1. Objetivo del proyecto

## 2. Librerías

## 3. Configuración de Pandas

## 4. Rutas

## 5. Funciones de lectura y transformación

## 6. Carga de datos

## 7. Validación de carga
    - shape
    - info()
    - nulos
    - duplicados

## 8. Transformaciones de negocio
    - Importe Ajustado
    - Factor
    - Corrección de tipos de documento
    - Otras reglas del ERP

## 9. Modelado
    - merge clientes
    - merge vendedores
    - merge artículos
    - merge margen

## 10. Feature Engineering
    - columnas nuevas
    - KPIs
    - métricas

## 11. Exportación
    - parquet
    - csv
    - excel

In [1]:
# 2. Librerias
# Importa el módulo OS (Operating System) de Python
#Importa Path de la librería pathlib. Sirve para gestionar rutas de archivos y carpetas de forma inteligente, moderna y visual.
#Importa pandas, una biblioteca de Python para el análisis y manipulación de datos

import os
from pathlib import Path
import pandas as pd


In [2]:
# 3. Configuracion de Pandas
# Muestra todas las columnas sin límite
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

#Si quiero desativar la opción de mostrar todas las columnas, puedo usar el siguiente comando:
#pd.reset_option('display.max_columns')
#pd.reset_option('display.max_rows')

In [3]:
# 4. Rutas

ruta_ventas_historicas = Path("../Ventas_Historico/Fuentes/Historico_Ventas")
ruta_margen_utilidad = Path("../Ventas_Historico/Fuentes/Historico_Margen")
ruta_maestros = Path("../Ventas_Historico/Fuentes/Maestros")

In [4]:
# 5. Funciones de lectura de archivos y transformacion

def leer_ventas(ruta):
    # Buscar dinámicamente todos los archivos xls que empiecen con "ventas_"
    rutas_archivos_ventas = list(ruta.glob("Ventas_*.xls"))

    #Lista para almacenar los DataFrames individuales
    lista_dataframes_ventas = []

    # Bucle para leer cada archivo encontrado de ventas y asignarle un numero que identifica de que archivo viene
    digito_archivo = 1
    for archivo in rutas_archivos_ventas:
        print(f"Leyendo de forma dinámica: {archivo.name}")

        # Leer el archivo actual y eliminar las primeras 18 filas. Ademas como tuve problemas por como pandas lee las fecha, le pido que a la columna "Fecha" me la devuelva
        # directamente como un str, y despues la voy a forzar a que tome el primer valor como dia.
        df = pd.read_excel(archivo, skiprows=18, engine="openpyxl", dtype={'Fecha': str})

        # La columna fecha es un str que viene del paso anterior, entonces le digo que tome el primer valor como dia, si no forzaba a la lectura como str pandas me convertia
        # los valores como fecha y a veces tomaba el primer valor como mes, y a veces como dia, al hacer el dayfirst ya estaba cambiado y no servia
        df["Fecha"] = pd.to_datetime(df["Fecha"], dayfirst=True, format="mixed", errors="coerce")

        partes_nombre = archivo.stem.split("_")  # Separa el nombre por guiones bajos
        
        # El valor del tipo de venta esta en la ultima posicion del nombre con -1 alcanzo ese valor
        tipo_venta = partes_nombre[-1]
        df["Tipo Venta"] = str(tipo_venta)  # Guarda 'blanco' o 'negro'
        lista_dataframes_ventas.append(df)

        #Asignacion de un digito en una nueva columna para identificar de que archivo salio el valor
        df["Archivo"] = digito_archivo
        digito_archivo = digito_archivo + 1
    
    # Concatenar todo solo si se encontraron archivos de ventas
    if lista_dataframes_ventas:
        df_ventas_total = pd.concat(lista_dataframes_ventas, axis=0, ignore_index=True)
        print("\n¡Todos los archivos de Ventas fueron concatenados con éxito!")
        #print(df_ventas_total.head(5))
    else:
        print("No se encontraron archivos de ventas en la carpeta.")

    return df_ventas_total

def leer_margenes(ruta):
    # Buscar dinámicamente todos los archivos xls que empiecen con "margen_"
    rutas_archivos_margen = list(ruta.glob("Margen_*.xls"))

    #Lista para almacenar los DataFrames individuales
    lista_dataframes_margen = []

    # Bucle para leer cada archivo encontrado de Margenes
    for archivo in rutas_archivos_margen:
        print(f"Leyendo de forma dinámica: {archivo.name}")

        # Leer el archivo actual y eliminar las primeras 5 filas  
        df = pd.read_excel(archivo, skiprows=5, engine="openpyxl")

        #Por ahora comento las transformaciones  
        # Opcional: Agregar columnas para saber de qué año y tipo proviene el dato
        # (Asumiendo nombres como 'ventas_2025_negro.csv')
        partes_nombre = archivo.stem.split("_")  # Separa el nombre por guiones bajos
        mes = partes_nombre[1]
        año = partes_nombre[2]
        tipo_venta = partes_nombre[3]
        df["Año"] = año
        df["Mes"] = mes
        df["Tipo Venta"] = tipo_venta

        lista_dataframes_margen.append(df)

    # Concatenar todo solo si se encontraron archivos de Margen
    if lista_dataframes_margen:
        df_margen_utilidad = pd.concat(lista_dataframes_margen, axis=0, ignore_index=True)
        print("\n¡Todos los archivos de Margenes fueron concatenados con éxito!")
        #print(df_margen_utilidad.head(5))
    else:
        print("No se encontraron archivos de Margen en la carpeta.")

    return df_margen_utilidad

def leer_articulos_clase(ruta):

    """ Lee el archivo "Maestro_Articulos_Clase.xls" y transforma el formato por bloques original en una tabla normalizada
    Parámetros:
    -----------

    ruta : pathlib.Path
        Carpeta donde se encuentra el archivo maestro

    Retorna
    -------

    pandas.DataFrame 
        DataFrame con los artículos, códigos de clase y descripción de clase
    """

    #El archivo que emite el sistema es un excel .xls con un formato por bloques y filas basura
    df_articulos_clase = pd.read_excel(ruta / "Maestro_Articulos_Clase.xls", engine="openpyxl")

    #Cada bloque comienza con una celda que identifica la clase
    filtro_clase = df_articulos_clase["Unnamed: 0"].str.startswith("Clas", na=False)

    df_articulos_clase.loc[filtro_clase, "Clase"] = (
        df_articulos_clase.loc[filtro_clase, "Unnamed: 0"]
    )

    #Aislado el dato de la clase se completan los valores
    df_articulos_clase["Clase"] = df_articulos_clase["Clase"].ffill()

    #Cada clase tiene una descripcion, se aisla el valor de descripcion de la clase
    df_articulos_clase["Descripcion"] = None
    filtro_clase = df_articulos_clase["Unnamed: 0"].str.startswith("Clase", na= False)
    df_articulos_clase.loc[filtro_clase, "Descripcion"] = df_articulos_clase.loc[filtro_clase, "Unnamed: 1"]

    #Una vez aislada la descripcion se completan los valores
    df_articulos_clase["Descripcion"] = df_articulos_clase["Descripcion"].ffill()

    #Como el archivo que exporta el sistema tiene filas basura es necesario eliminarlasy darle un nuevo formato
    #El archivo que exporta el sistema tiene 12 filas innecesarias
    df_articulos_clase.drop(df_articulos_clase.index[:12], inplace= True)
    df_articulos_clase.reset_index(drop = True, inplace = True)
    df_articulos_clase.columns = df_articulos_clase.iloc[0]
    df_articulos_clase = df_articulos_clase[1:].reset_index(drop=True)

    #Se acomodan los encabezados para darle forma al df
    df_articulos_clase = df_articulos_clase.dropna(subset=["Codigo"])

    #Se termina renombrando las celdas para que tengan un sentido
    df_articulos_clase.rename(
        columns={
            "Clase 1017": "Codigo Clase", "100 - EPICUREAN CHAMPAGNE - EPC - 40%" : "Descripcion Clase"
        },
        inplace= True
    )

    #Se realiza validacion para saber si funciono sin errores la funcion o si los archivos estan en la carpeta
    if df_articulos_clase.empty:
        raise ValueError("El DataFrame de artículos por clase quedó vacío.")
    else:
        print("Archivo de Articulos por cLases Listo")

    return df_articulos_clase

def leer_articulos_proveedor(ruta):

    """ Lee el archivo "Maestro_Articulos_Proveedor.xls" y transforma el formato por bloques original en una tabla normalizada
    Parámetros:
    -----------

    ruta : pathlib.Path
        Carpeta donde se encuentra el archivo maestro

    Retorna
    -------

    pandas.DataFrame 
        DataFrame con los artículos, códigos de clase y descripción de clase
    """
    #El archivo que emite el sistema es un excel .xls con un formato por bloques y filas basura
    df_articulos_proveedor = pd.read_excel(ruta / "Maestro_Articulos_Proveedor.xls", engine="openpyxl")

   #Cada bloque comienza con una celda que identifica al proveedor
    filtro_proveedor = df_articulos_proveedor["Unnamed: 0"].str.startswith("Prov", na=False)

    df_articulos_proveedor.loc[filtro_proveedor, "Proveedor"] = (
        df_articulos_proveedor.loc[filtro_proveedor, "Unnamed: 0"]
    )

    #Aislado el dato del proveedor se completan los valores
    df_articulos_proveedor["Proveedor"] = df_articulos_proveedor["Proveedor"].ffill()

    #Detecta descripciones
    df_articulos_proveedor["Descripcion"] = None
    filtro_proveedor = df_articulos_proveedor["Unnamed: 0"].str.startswith("Prove", na= False)
    df_articulos_proveedor.loc[filtro_proveedor, "Descripcion"] = df_articulos_proveedor.loc[filtro_proveedor, "Unnamed: 1"]

    #Completa descipciones
    df_articulos_proveedor["Descripcion"] = df_articulos_proveedor["Descripcion"].ffill()

    #Como el archivo que exporta el sistema tiene filas basura es necesario eliminarlas y darle un nuevo formato
    #El archivo que exporta el sistema tiene 14 filas innecesarias
    df_articulos_proveedor.drop(df_articulos_proveedor.index[:14], inplace= True)
    df_articulos_proveedor.reset_index(drop = True, inplace = True)
    df_articulos_proveedor.columns = df_articulos_proveedor.iloc[0]
    df_articulos_proveedor = df_articulos_proveedor[1:].reset_index(drop=True)

    #Se acomodan los encabezados para darle forma al df
    df_articulos_proveedor = df_articulos_proveedor.dropna(subset=["Codigo"])

    #Se termina renombrando las celdas para que tengan un sentido
    df_articulos_proveedor.rename(
        columns={
            "Proveedor 1": "Codigo Proveedor", "Patricio Prestamo" : "Descripcion Proveedor"
        },
        inplace= True
    )

    #Se realiza validacion para saber si funciono sin errores la funcion o si los archivos estan en la carpeta
    if df_articulos_proveedor.empty:
        raise ValueError("El DataFrame de artículos por proveedor quedó vacío.")
    else:
        print("Archivo de Articulos por Proveedor Listo")

    return df_articulos_proveedor

def leer_vendedores(ruta):
    df_vendedor = pd.read_excel(ruta / "Vendedores.xlsx", engine="openpyxl")
    return df_vendedor

def read_clients(path):
    df_clients = pd.read_excel(path / "Clientes.xls",skiprows=18, engine="openpyxl")
    return df_clients

In [5]:
# 6. Carga de Datos

df_ventas_total = leer_ventas(ruta_ventas_historicas)

df_margen_utilidad = leer_margenes(ruta_margen_utilidad)

df_articulos_clase = leer_articulos_clase(ruta_maestros)

df_articulos_proveedor = leer_articulos_proveedor(ruta_maestros)

df_vendedor = leer_vendedores(ruta_maestros)

df_clients = read_clients(ruta_maestros)

Leyendo de forma dinámica: Ventas_2025_B.xls
Leyendo de forma dinámica: Ventas_2025_N.xls
Leyendo de forma dinámica: Ventas_2026_06_B.xls
Leyendo de forma dinámica: Ventas_2026_06_N.xls
Leyendo de forma dinámica: Ventas_2026_07_B.xls
Leyendo de forma dinámica: Ventas_2026_07_N.xls
Leyendo de forma dinámica: Ventas_2026_B.xls
Leyendo de forma dinámica: Ventas_2026_N.xls

¡Todos los archivos de Ventas fueron concatenados con éxito!
Leyendo de forma dinámica: Margen_01_2025_B.xls
Leyendo de forma dinámica: Margen_01_2025_N.xls
Leyendo de forma dinámica: Margen_01_2026_B.xls
Leyendo de forma dinámica: Margen_01_2026_N.xls
Leyendo de forma dinámica: Margen_02_2025_B.xls
Leyendo de forma dinámica: Margen_02_2025_N.xls
Leyendo de forma dinámica: Margen_02_2026_B.xls
Leyendo de forma dinámica: Margen_02_2026_N.xls
Leyendo de forma dinámica: Margen_03_2025_B.xls
Leyendo de forma dinámica: Margen_03_2025_N.xls
Leyendo de forma dinámica: Margen_03_2026_B.xls
Leyendo de forma dinámica: Margen_03_2

In [6]:
# 7. Validación de carga

print(f"Ventas: {df_ventas_total.shape}")
print(f"Margen: {df_margen_utilidad.shape}")
#print(f"Clientes: {df_clientes.shape}")
print(f"Artículos Clase: {df_articulos_clase.shape}")
print(f"Artículos Proveedor: {df_articulos_proveedor.shape}")
print(f"Vendedores: {df_vendedor.shape}")

print("\nValores nulos:")
print(df_ventas_total.isna().sum())

print("\nDuplicados Ventas:")
print(df_ventas_total.duplicated().sum())

print("\nDuplicados Margenes:")
print(df_margen_utilidad.duplicated().sum())

print("\nDuplicados Articulos por Clase:")
print(df_articulos_clase.duplicated().sum())

print("\nDuplicados Articulos por Proveedor:")
print(df_articulos_proveedor.duplicated().sum())

print("\nTipos de datos:")
print(df_ventas_total.dtypes)

Ventas: (63936, 25)
Margen: (14573, 28)
Artículos Clase: (7605, 4)
Artículos Proveedor: (8546, 4)
Vendedores: (44, 4)

Valores nulos:
Codigo               0
Descripcion          0
Cantidad             0
Importe              0
Fecha                0
Doc                  0
Tipo                 0
Numero               0
xUnidad de Medida    0
-                    0
Cliente              0
Nombre               0
Razon Social         0
CUIT                 0
Documento            0
Vendedor             0
C.Postal             0
Localidad            0
Direccion            0
Estado Cta           0
P.Pago               0
Lista                0
Color                0
Tipo Venta           0
Archivo              0
dtype: int64

Duplicados Ventas:
27

Duplicados Margenes:
0

Duplicados Articulos por Clase:
0

Duplicados Articulos por Proveedor:
0

Tipos de datos:
Codigo                        int64
Descripcion                     str
Cantidad                    float64
Importe                     floa

In [7]:
duplicados = df_ventas_total[df_ventas_total.duplicated(keep=False)]

#duplicados.sort_values(
#    ["Numero", "Codigo", "Fecha"]
#)

8. Transformaciones de negocio

In [8]:
#Los campos importe y cantidad se encuentran como un valor positivo, aunque el tipo de movimiento haya sido una facturacion "FAC" o nota de debito "NDE", o una nota de 
#credito "NCR" por esta razon se crea una columna factor con un valor 1 y que si el valor de "Doc" es "NCR" le asigna -1. Luego se agrega la columna "Importe Ajustado" y 
#"Cantidad Ajustada" que calculan el valor con el signo correspondiente para hacer calculos directos
df_ventas_total["Factor"] = 1
df_ventas_total.loc[df_ventas_total["Doc"] == "NCR", "Factor"] = -1
df_ventas_total["Importe Ajustado"] = df_ventas_total["Importe"] * df_ventas_total["Factor"]
df_ventas_total["Cantidad Ajustada"] = df_ventas_total["Cantidad"] * df_ventas_total["Factor"]
df_ventas_total["Año"] = df_ventas_total["Fecha"].dt.year.astype(str)
df_ventas_total["Mes"] = df_ventas_total["Fecha"].dt.strftime("%m")

In [9]:
#Se elimina duplicados que vienen de diferentes archivos, pero se dejan los que vienen dentro de un mismo archivo
columnas_comparacion = df_ventas_total.columns.drop("Archivo")
duplicados = df_ventas_total[
    df_ventas_total.duplicated(
        subset=columnas_comparacion,
        keep=False
    )
].copy()
columnas_comparacion = columnas_comparacion.tolist()
grupos = duplicados.groupby(columnas_comparacion)
filas_a_eliminar = []
for nombre, grupo in grupos:
    if grupo["Archivo"].nunique() > 1:
        grupo = grupo.sort_values("Archivo")
        filas_a_eliminar.append(grupo.iloc[1:])
filas_a_eliminar = pd.concat(filas_a_eliminar)
df_ventas_total = df_ventas_total.drop(index=filas_a_eliminar.index)

In [10]:
#Se enriquese el df de ventas con informacion de los demas df.

df_ventas_total = df_ventas_total.drop(columns=["Descripcion"], errors="ignore")

columnas_prov = [col for col in df_articulos_proveedor.columns if col != "Descripcion"]
df_ventas_total = df_ventas_total.merge(
    df_articulos_proveedor[columnas_prov],
    on="Codigo",
    how="left"
)

df_ventas_total = df_ventas_total.merge(
    df_articulos_clase,
    on="Codigo",
    how="left"
)

df_ventas_total = df_ventas_total.merge(
    df_vendedor.rename(columns={"Codigo": "Codigo Vendedor",
                                "Nombre": "Nombre Vendedor"}),
    left_on="Vendedor",
    right_on="Codigo Vendedor",
    how="left"
).drop(columns=["Vendedor"])


df_ventas_total = df_ventas_total.merge(
    df_clients[["Codigo", "Cliente"]],
    left_on="Cliente",
    right_on="Codigo",
    how="left",
    suffixes=("", "_eliminar")  # Deja intacto el 'Codigo' de la izquierda y renombra el de la derecha
).drop(columns=["Cliente_eliminar", "Codigo_eliminar"])  # Borra el código de los clientes que no querías

df_ventas_total = df_ventas_total.merge(
    df_margen_utilidad[["Año", "Mes", "Codigo", "Tipo Venta", "Margen"]],
    left_on=["Año", "Mes", "Codigo", "Tipo Venta"],
    right_on=["Año", "Mes", "Codigo", "Tipo Venta"],
    how= "left"
)

In [11]:
df_ventas_total = df_ventas_total.drop(columns=[
    "xUnidad de Medida",
    "-",
    "Color"
    ])
df_ventas_total = df_ventas_total.rename(columns=
                                         {"Codigo":"Codigo Articulo",
                                          "Nombre": "Nombre Cliente",
                                          "Descripcion": "Descripcion Articulo",
                                          "Margen": "Margen Utilidad"
                                          })
                                          

In [12]:
df_ventas_total.tail(10)

,Codigo Articulo,Cantidad,Importe,Fecha,Doc,Tipo,Numero,Cliente,Nombre Cliente,Razon Social,CUIT,Documento,C.Postal,Localidad,Direccion,Estado Cta,P.Pago,Lista,Tipo Venta,Archivo,Factor,Importe Ajustado,Cantidad Ajustada,Año,Mes,Codigo Proveedor,Descripcion Proveedor,Descripcion Articulo,Codigo Clase,Descripcion Clase,Codigo Vendedor,Nombre Vendedor,Zona,Estado,Margen Utilidad
84285,5070,2.0,375400.0,2026-05-28,FAC,B,9000101373,1,Consumidor Final,Foresta BCH,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,375400.0,2.0,2026,05,Proveedor 1743,Fruto De La Confianz,Nuez Mariposa Extra Light 10kg SIN TACC,Clase 110,279 - FRUTAS SECAS A GRANEL,BCHE-C,Paulo,Bariloche,Normal,26.55
84286,5917,1.0,19000.0,2026-05-28,FAC,B,9000101373,1,Consumidor Final,Foresta BCH,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,19000.0,1.0,2026,05,Proveedor 1577,Fruggina,Jugo CAJA Fruggina 12u de 500c MANZANA,Clase 229,214 - JUGOS FRUTALES Y PULPAS,BCHE-C,Paulo,Bariloche,Normal,34.15
84287,6838,1.0,19000.0,2026-05-28,FAC,B,9000101373,1,Consumidor Final,Foresta BCH,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,19000.0,1.0,2026,05,Proveedor 1577,Fruggina,Jugo CAJA Fruggina 12u de 500c MANZ VER,Clase 229,214 - JUGOS FRUTALES Y PULPAS,BCHE-C,Paulo,Bariloche,Normal,34.15
84288,5918,1.0,19000.0,2026-05-28,FAC,B,9000101373,1,Consumidor Final,Foresta BCH,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,19000.0,1.0,2026,05,Proveedor 1577,Fruggina,Jugo CAJA Fruggina 12u de 500c PERA,Clase 229,214 - JUGOS FRUTALES Y PULPAS,BCHE-C,Paulo,Bariloche,Normal,34.15
84289,7399,1.0,42300.0,2026-05-29,FAC,B,9000101374,1,Consumidor Final,Clara Vending,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,42300.0,1.0,2026,05,Proveedor 1804,Las Perdices,Gin Perdix London Dry 750,Clase 22,222 - GIN,MG Salon,Marcas Garage,NaN,Normal,28.38
84290,6376,2.0,24800.0,2026-05-29,FAC,B,9000101375,1,Consumidor Final,NY Cookies Sebastian,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,24800.0,2.0,2026,05,Proveedor 1151,Alyser X Fracc,Arandanos Deshidratados 1kg,Clase 78,263 - FRUTAS SECAS Y DESHIDRATADAS FRACCIONADAS,BCHE-C E,Gonzalo,Bariloche,Normal,29.66
84291,6510,1.0,35400.0,2026-05-29,FAC,B,9000101375,1,Consumidor Final,NY Cookies Sebastian,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,35400.0,1.0,2026,05,Proveedor 2031,Fenix,Cob. S/Amarg 62% - ECUADOR x1Kg FNX,Clase 34,306 - COBERTURAS - ALTO % CACAO SIN TACC,BCHE-C E,Gonzalo,Bariloche,Normal,26.50
84292,5274,1.0,22800.0,2026-05-29,FAC,B,9000101375,1,Consumidor Final,NY Cookies Sebastian,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,22800.0,1.0,2026,05,Proveedor 21,Alyser S.A.,Cob. S/Amarga - PICOS DEL SUR 1 kg,Clase 922,308 - COBERTURA DE CHOCOLATE - PICOS DEL SUR,BCHE-C E,Gonzalo,Bariloche,Normal,26.65
84293,6425,1.0,15800.0,2026-05-29,FAC,B,9000101375,1,Consumidor Final,NY Cookies Sebastian,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,15800.0,1.0,2026,05,Proveedor 1804,Las Perdices,Bag Pouch - Chac Chac Cab Franc x 2250cc,Clase 202,117 - LAS PERDICES - BAG IN BOX 50%,BCHE-C E,Gonzalo,Bariloche,Normal,28.52
84294,6425,1.0,15800.0,2026-05-29,FAC,B,9000101375,1,Consumidor Final,NY Cookies Sebastian,0,S/D,1,Indefinido,Uiutg,No Posee,0,Bche Distr,N,8,1,15800.0,1.0,2026,05,Proveedor 2090,Almacen De Vinos,Bag Pouch - Chac Chac Cab Franc x 2250cc,Clase 202,117 - LAS PERDICES - BAG IN BOX 50%,BCHE-C E,Gonzalo,Bariloche,Normal,28.52


In [13]:
cantidad_nan = df_ventas_total['Margen Utilidad'].isna().sum()

df_ventas_nan = df_ventas_total[df_ventas_total['Margen Utilidad'].isna()]

print(f"Total de filas vacías en Margen: {cantidad_nan}")

df_ventas_nan

Total de filas vacías en Margen: 780


,Codigo Articulo,Cantidad,Importe,Fecha,Doc,Tipo,Numero,Cliente,Nombre Cliente,Razon Social,CUIT,Documento,C.Postal,Localidad,Direccion,Estado Cta,P.Pago,Lista,Tipo Venta,Archivo,Factor,Importe Ajustado,Cantidad Ajustada,Año,Mes,Codigo Proveedor,Descripcion Proveedor,Descripcion Articulo,Codigo Clase,Descripcion Clase,Codigo Vendedor,Nombre Vendedor,Zona,Estado,Margen Utilidad
259,6678,1.000,8105.79,2025-01-02,FAC,A,1100004019,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,1,8105.79,1.000,2025,01,Proveedor 1672,Enrique Tomas,Caja Cartón Porta JAMON,Clase 27,250 - ACCESORIOS - ENRIQUE TOMAS,BCHE-C,Paulo,Bariloche,Normal,NaN
260,6820,1.000,0.00,2025-01-02,FAC,A,1100004019,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,1,0.00,1.000,2025,01,Proveedor 1672,Enrique Tomas,HELADERA - E. TOMAS (Comodato),Clase 214,251 - EMBUTIDOS Y JAMONES LONCHEADOS,BCHE-C,Paulo,Bariloche,Normal,NaN
1002,6678,1.000,8105.79,2025-01-06,NCR,A,500005282,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,-1,-8105.79,-1.000,2025,01,Proveedor 1672,Enrique Tomas,Caja Cartón Porta JAMON,Clase 27,250 - ACCESORIOS - ENRIQUE TOMAS,BCHE-C,Paulo,Bariloche,Normal,NaN
1199,6678,1.000,0.00,2025-01-07,FAC,A,1100004043,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,1,0.00,1.000,2025,01,Proveedor 1672,Enrique Tomas,Caja Cartón Porta JAMON,Clase 27,250 - ACCESORIOS - ENRIQUE TOMAS,BCHE-C,Paulo,Bariloche,Normal,NaN
1200,6820,1.000,0.00,2025-01-07,FAC,A,1100004043,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,1,0.00,1.000,2025,01,Proveedor 1672,Enrique Tomas,HELADERA - E. TOMAS (Comodato),Clase 214,251 - EMBUTIDOS Y JAMONES LONCHEADOS,BCHE-C,Paulo,Bariloche,Normal,NaN
1211,6678,1.000,0.00,2025-01-07,FAC,A,1100004044,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,1,0.00,1.000,2025,01,Proveedor 1672,Enrique Tomas,Caja Cartón Porta JAMON,Clase 27,250 - ACCESORIOS - ENRIQUE TOMAS,BCHE-C,Paulo,Bariloche,Normal,NaN
1212,6820,1.000,0.00,2025-01-07,FAC,A,1100004044,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,1,0.00,1.000,2025,01,Proveedor 1672,Enrique Tomas,HELADERA - E. TOMAS (Comodato),Clase 214,251 - EMBUTIDOS Y JAMONES LONCHEADOS,BCHE-C,Paulo,Bariloche,Normal,NaN
1232,6678,1.000,8105.79,2025-01-07,NCR,A,1100000102,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,-1,-8105.79,-1.000,2025,01,Proveedor 1672,Enrique Tomas,Caja Cartón Porta JAMON,Clase 27,250 - ACCESORIOS - ENRIQUE TOMAS,BCHE-C,Paulo,Bariloche,Normal,NaN
1233,6820,1.000,0.00,2025-01-07,NCR,A,1100000102,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,-1,-0.00,-1.000,2025,01,Proveedor 1672,Enrique Tomas,HELADERA - E. TOMAS (Comodato),Clase 214,251 - EMBUTIDOS Y JAMONES LONCHEADOS,BCHE-C,Paulo,Bariloche,Normal,NaN
1244,6678,1.000,0.00,2025-01-07,NCR,A,1100000103,40268,Tannini,TANNINI S. A. S.,30718625498,S/D,8400,S.C. de Bariloche,Palacios 149,Operativa,4001,Bche Distr,B,1,-1,-0.00,-1.000,2025,01,Proveedor 1672,Enrique Tomas,Caja Cartón Porta JAMON,Clase 27,250 - ACCESORIOS - ENRIQUE TOMAS,BCHE-C,Paulo,Bariloche,Normal,NaN
